# 🛡️ BlockAds Filter — Cloud Runner & Optimizer on Google Colab

Notebook này chạy trực tiếp trên máy ảo **Google Colab**, tận dụng băng thông Google Cloud để:
1. Quét và loại bỏ các tên miền chết (NXDOMAIN).
2. Rebuild và tải trực tiếp các gói nhị phân (.zip chứa .trie, .bloom, .css) lên **Cloudflare R2** mà không tốn tài nguyên máy cá nhân.

### Bước 1: Chuẩn bị mã nguồn từ GitHub

In [ ]:
# @title 1. Cập nhật mã nguồn mới nhất
import os
if not os.path.exists('/content/blockads_filter_bin'):
    !git clone https://github.com/pass-with-high-score/blockads_filter_bin.git /content/blockads_filter_bin
%cd /content/blockads_filter_bin
!git pull origin main

### Bước 2: Quét & lọc một bộ lọc (URL tùy ý)
Bạn có thể nhập bất kỳ URL danh sách chặn nào vào ô bên dưới.

In [ ]:
# @title 2. Quét & Lọc theo URL
filter_url = "https://www.zoso.ro/pages/rolist.txt" # @param {type:"string"}
concurrency = 600 # @param {type:"integer"}
timeout_sec = 2.0 # @param {type:"number"}
output_file = "cleaned_filter.txt" # @param {type:"string"}

!python3 scripts/colab_clean_filter.py \
    --url "$filter_url" \
    --concurrency $concurrency \
    --timeout $timeout_sec \
    --output "$output_file"

### Bước 3: Cài đặt Bun Runtime (cho Database & Rebuild mode)

In [ ]:
# @title 3. Cài đặt Bun & dependencies
!curl -fsSL https://bun.sh/install | bash
import os
os.environ["PATH"] = f"{os.environ['HOME']}/.bun/bin:{os.environ['PATH']}"
!bun install

### Bước 4: Quét toàn bộ bộ lọc trong Database
> **Mẹo:** Bạn có thể lưu `DATABASE_URL` vào mục **Secrets (🔑 chìa khóa ở thanh bên trái)** trong Colab.

In [ ]:
# @title 4. Chạy quét từ Database
import getpass, os
try:
    from google.colab import userdata
    db_url = userdata.get('DATABASE_URL')
except Exception:
    db_url = os.environ.get('DATABASE_URL', '')

if not db_url:
    db_url = getpass.getpass("Nhập DATABASE_URL: ")

os.environ['DATABASE_URL'] = db_url

# Số lượng bộ lọc cần quét (chọn --limit 10 hoặc --all để quét tất cả)
limit = 10 # @param {type:"integer"}
!bun run scripts/clean-filters-db.ts --limit $limit

### Bước 5: Rebuild toàn bộ và Upload trực tiếp lên Cloudflare R2 từ Colab 🚀
Chạy toàn bộ quy trình biên dịch nhị phân và upload lên Cloudflare R2 CDN.

> **Cần các biến:** `DATABASE_URL`, `R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY`, `R2_BUCKET_NAME`, `R2_PUBLIC_URL` (thêm vào mục **Secrets 🔑**).

In [ ]:
# @title 5. Chạy Rebuild All & Upload Cloudflare R2
import os
try:
    from google.colab import userdata
    for key in ["DATABASE_URL", "R2_ACCOUNT_ID", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET_NAME", "R2_PUBLIC_URL"]:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except Exception:
    pass

!bun run scripts/rebuild-all.ts